In [3]:
# -*- coding: utf-8 -*-
"""
Scan CI YAML files for instrumentation-testing signals and output:
filename, full_name, ci_platform, instru_t_ci_signal, confidence, confidence_reason,
execution_environment, test_invocation, flutter_integ_t_signal, flutter_integ_t_d

INCLUSIVITY UPGRADES (CI YAML ONLY, schema unchanged):
- Detects compact emulator action (malinskiy/action-android/emulator-run-cmd).
- More robust YAML normalization (handles |, >, |- scalars, list dashes).
- Shell-robust emulator/ADB patterns (env/sudo/bash/pwsh/cd && wrappers).
- Accepts common key variants (api_level, avd-name, -noaudio) as WEAK hints.
- Broader GMD inference inside YAML (deviceGroups, testOptions devices/groups).
- Recognize gradle/gradle-build-action as "Gradle present" (for fallback).
- Extra CI vendor clues (CircleCI helper, Docker Android SDK images, Azure task).

NOTE: Paths are set for Windows; change CONFIG_DIR / TESTS_DIR / OUTPUT_CSV for your environment.
"""

import re
import pandas as pd
from pathlib import Path
from typing import List, Pattern, Tuple, Dict, Any, Iterable

# === CONFIG ===
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files")
TESTS_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Test_Files")
OUTPUT_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_YML_FilesV7.0.csv")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
TESTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# === Helpers (naming conventions preserved) ===
def extract_full_name_from_file(filename: str) -> str:
    fname = filename.lower()
    if "__" in fname:
        return fname.split("__", 1)[0]
    return Path(fname).stem

def extract_ci_platform(filename: str) -> str:
    """owner.repo__github++file.yml  -> github"""
    fname = filename.lower()
    if "__" in fname and "++" in fname:
        return fname.split("__", 1)[1].split("++", 1)[0]
    return ""

def build_test_presence_index(tests_dir: Path) -> set[str]:
    present = set()
    for f in tests_dir.glob("*"):
        if not f.is_file():
            continue
        name = f.name.lower()
        if "__" in name:
            repo_key = name.split("__", 1)[0]
            present.add(repo_key)
    return present

ANDROIDTEST_PRESENT = build_test_presence_index(TESTS_DIR)

def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: Iterable[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

# Strip comments (#, //, Windows batch comments, PowerShell ::)
COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

def normalize_block_keys(text: str) -> str:
    """
    Normalize YAML blocks so signals in run/script/command are visible:
    - Inline:  "run: ./gradlew ..."    -> "./gradlew ..."
    - Multiline: "run: |" / "run: >"   -> drop key line, keep content
    - Remove YAML list dashes that break ^ anchors
    """
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\||>|\|\-)\s*(.+)$', r'\2', text)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(\||>|\|\-)\s*$', '', text)
    text = re.sub(r'(?m)^\s*-\s*', '', text)
    return text

# Gradle command prefix normalizer (env/sudo/bash/cd && ./gradlew)
GRADLE_PREFIX = (
    r'^\s*'
    r'(?:\S+=\S+\s+)*'
    r'(?:sudo\s+)?'
    r'(?:(?:bash|sh)\s+-c[l]?\s+[\'"]?)?'
    r'(?:[^#\n;]*?&&\s+)?'
    r'(?:cd\s+\S+\s+&&\s+)?'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?'
)

GRADLE_ANYWHERE = r'(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*'
NON_TEST_PREFIX = r'(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)'

# --- Shell-robust prefixes for emulator/adb lines ---
SHELL_PREFIX = r'(?:\S+=\S+\s+)*(?:sudo\s+)?(?:(?:bash|sh|pwsh|powershell)\s+-c\s+[\'"]?)?(?:[^#\n;]*?&&\s+)?'
EMULATOR_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}\bemulator\b[^\n]*(?:-avd\s+\S+|@\S+)'
ADB_WAIT_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}adb\s+wait[- ]?for[- ]?device\b'
ADB_SERIAL_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}adb\s+-s\s+(?:emulator-\d+|localhost:\d+|127\.0\.0\.1:\d+)'

# === Device/Trigger sources ===

DEVICE_SOURCES = [
    # Strong REAL device (only)
    ("Real_Device", "adb -s <serial> (physical)", [r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b']),

    # Emulator/AVD strong hints (shell-robust)
    ("Emulator", "adb -s emulator-serial", [ADB_SERIAL_LINE]),
    ("Emulator", "adb wait-for-device",   [ADB_WAIT_LINE]),
    ("Emulator", "emulator -avd/@",       [EMULATOR_LINE]),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^\s*(?:\./)?android-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh", [r'(?m)^\s*start-emulator\.sh\b']),
    ("Emulator", "android create avd", [r'\bandroid\b[^\n]*\bcreate\s+avd\b']),
    ("Emulator", "circle-android wait-for-boot",[r'(?m)^\s*circle-android\s+wait-for-boot\b']),
    ("Emulator", "reactivecircus runner", [r'(?mi)\buses\s*:\s*reactivecircus/android-emulator-runner@[\w\.\-]+']),
    # NEW: Compact GH action
    ("Emulator", "malinskiy runner", [r'(?mi)\buses\s*:\s*malinskiy/action-android/emulator-run-cmd@[\w\.\-]+']),

    # sys-img / avdmanager / sdkmanager
    ("Emulator", "sys-img component", [
        r'(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
        r'(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "avdmanager", [r'(?m)^\s*\S*avdmanager\b']),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r'^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
    ]),

    # WEAK hints / common keys (kept weak by filter)
    ("Emulator", "headless flag", [r'(?mi)\b-no-?audio\b', r'(?mi)\b-no-window\b', r'(?mi)\b-no-boot-anim\b']),
    ("Emulator", "avd-name",      [r'(?mi)^\s*avd[-_ ]?name\s*:\s*\S+']),
    ("Emulator", "api-level",     [r'(?mi)\bapi[-_ ]?level\s*:\s*\d{2,}|\bapi_level\s*:\s*\d{2,}']),
    ("Emulator", "abi/arch",      [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image",  [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name",   [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),

    # GMD (Gradle Managed Devices) cues visible in YAML (indirect)
    ("GMD_Intent", "managedDevices DSL",       [r'\bmanageddevices?\b']),
    ("GMD_Intent", "ManagedVirtualDevice DSL", [r'\bmanagedvirtualdevice\b|\bcom\.android\.build\.api\.dsl\.ManagedVirtualDevice\b']),
    ("GMD_Intent", "GMD task mentions",        [r'\bmanageddevice\w*androidtest\b']),
    ("GMD_Intent", "GHA gradle args for GMD",  [
        r'(?m)^\s*arguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
        r'(?m)^\s*tasks?\s*:\s*[:\w-]*manageddevice\w*androidtest\b'
    ]),
    # NEW: groups/testOptions mentions seen in YAML contexts/templates
    ("GMD_Intent", "GMD groups mention", [
        r'(?mi)\bdevicegroups?\b',
        r'(?mi)\btestoptions\b[^\n]*\b(devices|groups)\b'
    ]),

    # Third-party device labs
    ("Third_Party_Lab", "gcloud firebase", [r'(?m)^\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",        [r'(?m)^\s*saucectl(\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack", [r'\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",  [r'(?m)^\s*appcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",   [r'(?m)^\s*maestro\s+cloud\b']),

    # Extra CI vendor clues (still YAML-level)
    ("Emulator", "circleci android", [r'(?mi)^\s*circleci\s+android\s+wait-for-boot\b']),
    ("Emulator", "docker android sdk image", [
        r'(?mi)\bimage\s*:\s*(?:reactivecircus|cirrusci|budtmo)/android[-\w:]*'
    ]),
    ("Emulator", "azure android sdk setup", [r'(?mi)\btask\s*:\s*AndroidToolInstaller@']),
]
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]

# --- Trigger (test invocation) patterns ---

# PRIMARY (line-anchored / gradle-prefixed)
TRIGGER_SOURCES_PRIMARY = [
    ("Gradle",  "connectedAndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedandroidtest\b[^\n\r]*']),
    ("Gradle",  "connected.*Android.*",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connected[a-z0-9:._-]*android[a-z0-9:._-]*test\b[^\n\r]*']),
    ("Gradle",  "connectedCheck",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedcheck\b[^\n\r]*']),

    ("Gradle",  "cAT shorthand",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*cAT\b[^\n\r]*']),

    ("Gradle",  "deviceCheck",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b[^\n\r]*']),
    ("Gradle",  "managedDevice AndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b[^\n\r]*']),
    ("Gradle",  "variant/device AndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b[^\n\r]*']),

    ("Gradle",  "Spoon",    [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\bspoon(?:\w*androidtest)?\b']),
    ("Gradle",  "Marathon", [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\bmarathon(?:\w*androidtest)?\b']),

    ("ADB",     "am instrument", [r'(?mi)^[^\n]*\bam\s+instrument\b']),

    ("Third_Party_Lab", "gcloud firebase", [r'(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "flank",            [r'(?mi)^[^\n]*\bflank\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",         [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run",    [r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b']),

    ("Flutter", "flutter drive",                       [r'(?mi)^[^\n]*\bflutter\s+drive\b']),
    ("Flutter", "flutter test (integration_test)",     [r'(?mi)^[^\n]*\bflutter\s+test\b[^\n]*\bintegration_test\b']),
    ("Flutter", "dart test (integration_test)",        [r'(?mi)^[^\n]*\bdart\s+test\b[^\n]*\bintegration_test\b']),
]

# MUST-FIX ADDITIONS: Baseline Profile triggers (primary)
TRIGGER_SOURCES_PRIMARY += [
    ("Gradle", "generateBaselineProfile",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*generatebaselineprofile\b[^\n\r]*']),
    ("Gradle", "collectBaselineProfile",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*collectbaselineprofile\b[^\n\r]*']),
    ("Gradle", "connectedBenchmarkAndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedbenchmarkandroidtest\b[^\n\r]*']),
]

# ANYWHERE (fallback on the whole job text)
TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b']),
    ("Gradle", "connectedAndroidTest (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bconnectedandroidtest\b']),
    ("Gradle", "connectedCheck (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\b(?:[:\w-]+:)*connectedcheck\b']),
    ("Gradle", "managedDevice AndroidTest (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),
    ("Gradle", "variant/device AndroidTest (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b']),
    ("Gradle", "Spoon (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bmarathon(?:\w*androidtest)?\b']),
    ("Gradle", "cAT shorthand (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\b(?:[:\w-]+:)*cAT\b']),
]

# MUST-FIX ADDITIONS: Baseline Profile triggers (anywhere)
TRIGGER_SOURCES_ANYWHERE += [
    ("Gradle", "generateBaselineProfile (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bgeneratebaselineprofile\b']),
    ("Gradle", "collectBaselineProfile (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bcollectbaselineprofile\b']),
    ("Gradle", "connectedBenchmarkAndroidTest (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bconnectedbenchmarkandroidtest\b']),
]

TRIGGER_PATTERNS_PRIMARY  = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_PRIMARY]
TRIGGER_PATTERNS_ANYWHERE = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_ANYWHERE]

# GitHub Actions gradle inputs, including script:
GHA_GRADLE_INPUTS = compile_any([
    r'(?mi)^\s*arguments\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connected(android)?test\b',
    r'(?mi)^\s*arguments\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*arguments\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*arguments\s*:\s*[^$`\n]*\b[\w:-]*androidtest\b',
    r'(?mi)^\s*tasks?\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connected(android)?test\b',
    r'(?mi)^\s*tasks?\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*tasks?\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*tasks?\s*:\s*[^$`\n]*\b[\w:-]*androidtest\b',
    # cAT in gradle-action inputs
    r'(?mi)^\s*arguments\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*cAT\b',
    r'(?mi)^\s*tasks?\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*cAT\b',
    # parse action "script:" inputs
    r'(?mi)^\s*script\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connected(android)?test\b',
    r'(?mi)^\s*script\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*script\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*cAT\b',
])

# Strong vs weak device labels
EMULATOR_STRONG_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner", "avdmanager",
    "sdkmanager system-images/emulator", "adb -s emulator-serial", "adb wait-for-device",
    "android create avd", "malinskiy runner",
}
REAL_DEVICE_STRONG_LABELS = {"adb -s <serial> (physical)"}
REAL_DEVICE_GENERIC_ADB = set()
STRONG_DEVICE_LABELS = EMULATOR_STRONG_LABELS | REAL_DEVICE_STRONG_LABELS
WEAK_DEVICE_LABELS = {"api-level", "api (compact)", "tag (compact)", "abi (compact)", "cmd (compact)",
                      "abi/arch", "target image", "device name", "headless flag", "avd-name"}

# --- Flutter device extraction helpers (unchanged) ---
FLUTTER_CMD_LINE_RE = re.compile(r'(?mi)^\s*flutter\s+(?:drive|test)\b[^\n]*')
FLUTTER_DEVICE_FLAG_RE = re.compile(r'(?i)\s+-d\s+(?P<dev>"[^"]+"|\'[^\']+\'|\S+)')
LINUX_HEADLESS_HINTS_RE = re.compile(r'(?mi)^\s*(xvfb-run|export\s+DISPLAY=|sudo\s+Xvfb)\b')
IOS_SIM_HINTS = compile_any([
    r'uses:\s*futureware-tech/simulator-action@',
    r'\bxcrun\s+simctl\b',
    r'\biphonesimulator\b',
    r'\bdestination\b[^\n]*platform=iOS',
    r'(?mi)^\s*model\s*:\s*["\']?\s*(iphone|ipad)\b',
])

# ---------- job splitter (heuristic) ----------
JOBS_ANCHOR_RE = re.compile(r'(?m)^(?P<indent>\s*)jobs\s*:\s*$')
ANY_KEY_RE     = re.compile(r'(?m)^(?P<indent>\s*)(?P<name>[\w-]+)\s*:\s*$')

def split_jobs_blocks(raw: str) -> List[Tuple[str, str]]:
    m = JOBS_ANCHOR_RE.search(raw)
    if not m:
        return [("__whole__", raw)]
    jobs_indent = len(m.group("indent"))
    lines = raw.splitlines(True)
    start_idx = raw[:m.end()].count("\n")
    candidates = []
    for i in range(start_idx, len(lines)):
        lm = ANY_KEY_RE.match(lines[i])
        if not lm:
            continue
        indent = len(lm.group("indent"))
        if indent > jobs_indent:
            candidates.append((i, indent, lm.group("name")))
    if not candidates:
        return [("__whole__", raw)]
    min_indent = min(indent for _, indent, _ in candidates)
    job_headers = [(i, name) for (i, indent, name) in candidates if indent == min_indent]
    if not job_headers:
        return [("__whole__", raw)]
    blocks = []
    header_indices = [i for i, _ in job_headers] + [len(lines)]
    for idx in range(len(job_headers)):
        i, name = job_headers[idx]
        j = header_indices[idx + 1]
        block_text = "".join(lines[i:j])
        blocks.append((name, block_text))
    return blocks

def collect_hits_with_groups(patterns: List[Tuple[str, str, List[Pattern]]], text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl); groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

def filter_weak_device_hints(labels, groups):
    if not (set(labels) & STRONG_DEVICE_LABELS):
        labels = [l for l in labels if l not in WEAK_DEVICE_LABELS]
        if not labels:
            groups = []
    return labels, groups

def reconcile_emulator_vs_real(labels, groups):
    lbls = set(labels)
    if lbls & EMULATOR_STRONG_LABELS:
        lbls -= REAL_DEVICE_GENERIC_ADB
        labels = [l for l in labels if l in lbls]
        if "Real_Device" in groups:
            has_real_after = bool(set(labels) & REAL_DEVICE_STRONG_LABELS)
            if not has_real_after:
                groups = [g for g in groups if g != "Real_Device"]
    return labels, groups

# === Main scan (schema unchanged) ===
rows: List[Dict[str, Any]] = []

# Recognize gradle-build-action as "Gradle present" too
GRADLE_BUILD_ACTION_RE = re.compile(r'(?mi)\buses\s*:\s*(gradle/gradle-build-action|gradle/actions/setup-gradle)@')

for f in sorted(CONFIG_DIR.iterdir()):
    if not f.is_file():
        continue
    if f.suffix.lower() not in (".yml", ".yaml"):
        continue

    filename = f.name
    full_name = extract_full_name_from_file(filename)
    ci_platform = extract_ci_platform(filename)

    try:
        raw = f.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        raw = ""

    # per-file aggregates
    file_trigger_labels: List[str] = []
    file_device_labels:  List[str] = []
    file_trigger_groups: List[str] = []
    file_device_groups:  List[str] = []
    file_has_test_trigger = False
    file_has_device_with_group = False
    file_gradle_present_any = False
    file_instru_signal_any = False
    file_flutter_devices: List[str] = []

    for job_name, job_raw in split_jobs_blocks(raw):
        content = strip_comments(job_raw)
        content = normalize_block_keys(content)

        # Device & trigger detection (prefix-first)
        dev_labels, dev_groups = collect_hits_with_groups(DEVICE_PATTERNS, content.lower())
        dev_labels, dev_groups = filter_weak_device_hints(dev_labels, dev_groups)
        dev_labels, dev_groups = reconcile_emulator_vs_real(dev_labels, dev_groups)

        trig_labels, trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, content.lower())

        # gradle-build-action inputs + "script:" hints
        if any_match(GHA_GRADLE_INPUTS, content):
            trig_labels = unique_preserve(trig_labels + ["gha gradle inputs/script"])
            trig_groups = unique_preserve(trig_groups + ["Gradle"])

        # Fallback "anywhere" scan if needed
        if not trig_labels:
            fallback = strip_comments(job_raw)
            fallback = normalize_block_keys(fallback)
            fallback = re.sub(r'(?m)^\s*sudo\s+', '', fallback)

            if not dev_labels:
                fb_dev_labels, fb_dev_groups = collect_hits_with_groups(DEVICE_PATTERNS, fallback.lower())
                fb_dev_labels, fb_dev_groups = filter_weak_device_hints(fb_dev_labels, fb_dev_groups)
                fb_dev_labels, fb_dev_groups = reconcile_emulator_vs_real(fb_dev_labels, fb_dev_groups)
                if fb_dev_labels or fb_dev_groups:
                    dev_labels  = unique_preserve(dev_labels  + fb_dev_labels)
                    dev_groups  = unique_preserve(dev_groups  + fb_dev_groups)

            fb_trig_labels, fb_trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, fallback.lower())

            if any_match(GHA_GRADLE_INPUTS, fallback):
                fb_trig_labels = unique_preserve(fb_trig_labels + ["gha gradle inputs/script"])
                fb_trig_groups = unique_preserve(fb_trig_groups + ["Gradle"])

            if fb_trig_labels:
                trig_labels = unique_preserve(trig_labels + fb_trig_labels)
                trig_groups = unique_preserve(trig_groups + fb_trig_groups)

        has_device_setup = bool(dev_labels)
        has_test_trigger = bool(trig_labels)
        gradle_present   = bool(re.search(GRADLE_ANYWHERE, content) or GRADLE_BUILD_ACTION_RE.search(content))

        # --- Flutter device inference per job (unchanged logic) ---
        if "Flutter" in trig_groups:
            found = []
            for m in FLUTTER_CMD_LINE_RE.finditer(content):
                line = m.group(0)
                d = FLUTTER_DEVICE_FLAG_RE.search(line)
                if d:
                    plat = d.group("dev").strip('"\'')
                    t = plat.lower()
                    if (t == "android" or t.startswith("emulator-") or
                        "sdk gphone" in t or "android sdk built for" in t or "pixel " in t):
                        found.append("android")
                    elif t in {"linux","macos","windows"}:
                        found.append(t)
                    elif t in {"ios","iphone","ipad","iphone simulator"}:
                        found.append("ios")
                    elif t in {"web","web-server","chrome","edge","firefox","safari"}:
                        found.append("web")
            if not found and any(p.search(job_raw) for p in IOS_SIM_HINTS):
                found.append("ios")
            if not found and (set(dev_labels) & EMULATOR_STRONG_LABELS or "Emulator" in dev_groups or "GMD_Intent" in dev_groups):
                found.append("android")
            if not found and LINUX_HEADLESS_HINTS_RE.search(content):
                found.append("linux")
            for plat in found:
                if plat and plat not in file_flutter_devices:
                    file_flutter_devices.append(plat)

        # --- Per-job decisive evidence flags ---
        has_emulator_group = any(g in {"Emulator", "GMD_Intent", "Third_Party_Lab"} for g in dev_groups)
        has_real_device_strong = any(lbl in REAL_DEVICE_STRONG_LABELS for lbl in dev_labels)

        instru_t_ci_signal_job = bool(
            has_test_trigger or (has_device_setup and (has_emulator_group or has_real_device_strong))
        )
        file_instru_signal_any = file_instru_signal_any or instru_t_ci_signal_job

        # aggregate per-file
        file_device_labels  = unique_preserve(file_device_labels  + dev_labels)
        file_device_groups  = unique_preserve(file_device_groups  + dev_groups)
        file_trigger_labels = unique_preserve(file_trigger_labels + trig_labels)
        file_trigger_groups = unique_preserve(file_trigger_groups + trig_groups)
        file_has_test_trigger = file_has_test_trigger or has_test_trigger

        if has_device_setup and (has_emulator_group or has_real_device_strong):
            file_has_device_with_group = True

        file_gradle_present_any = file_gradle_present_any or gradle_present

    # Optional fallback mapping: device present + any Gradle but no trigger groups -> Gradle
    if not file_trigger_groups and file_has_device_with_group and file_gradle_present_any:
        file_trigger_groups.append("Gradle")
        file_trigger_labels.append("fallback-gradle-with-device")

    # Confidence & reasons (unchanged schema)
    reasons = []
    if file_has_test_trigger or "fallback-gradle-with-device" in file_trigger_labels:
        reasons.append("test_trigger: " + ", ".join([l for l in file_trigger_labels if l != "fallback-gradle-with-device"]))
    if file_has_device_with_group:
        msg = "device_setup: " + ", ".join(file_device_labels)
        if file_gradle_present_any:
            msg += "; gradle present"
        reasons.append(msg)
    reason = " | ".join(r for r in reasons if r)

    confidence = ""
    if (file_has_test_trigger or "fallback-gradle-with-device" in file_trigger_labels) and file_has_device_with_group:
        confidence = "high"
    elif file_has_test_trigger or file_has_device_with_group:
        confidence = "medium"

    # AndroidTest presence boost
    if file_instru_signal_any and full_name in ANDROIDTEST_PRESENT:
        if confidence == "":
            confidence = "medium"
        elif confidence == "medium":
            confidence = "high"
        reason = (reason + " + boosted (AndroidTest present)").strip()

    # Derived style fields (unchanged)
    def map_execution_env(groups: List[str], labels: List[str]) -> str:
        """
        Map detected device groups/labels to a human-friendly execution environment.
        New granularity for emulator-based runs:
        - Generic_reactivecircus  -> reactivecircus/android-emulator-runner
        - Generic_Malinskiy       -> malinskiy/action-android/emulator-run-cmd
        - Generic_DIY             -> other emulator DIY setups (sdkmanager/avdmanager/adb/emulator lines, etc.)
        """
        s_groups = set(groups)
        s_labels = set(labels)

        # Highest-priority buckets (unchanged semantics)
        if "GMD_Intent" in s_groups:
            return "GMD_intent"
        if "Third_Party_Lab" in s_groups:
            return "Third Party"
        if "Real_Device" in s_groups:
            return "Real Device"

        # Emulator family with finer labeling
        if "Emulator" in s_groups:
            if "reactivecircus runner" in s_labels:
                return "Generic_reactivecircus"
            if "malinskiy runner" in s_labels:
                return "Generic_Malinskiy"
            return "Generic_DIY"

        return "Unknown"

    def map_test_invocation(groups: List[str]) -> str:
        s = set(groups)
        if "Third_Party_Lab" in s: return "3P CLIs"
        if "ADB" in s: return "ADB"
        if "Gradle" in s: return "Gradle"
        return "Unknown"

    rows.append({
        "filename": filename,
        "full_name": full_name,
        "ci_platform": ci_platform,
        "instru_t_ci_signal": bool(file_instru_signal_any),
        "confidence": confidence if file_instru_signal_any else "",
        "confidence_reason": reason if file_instru_signal_any else "",
        "execution_environment": map_execution_env(file_device_groups, file_device_labels),
        "test_invocation": map_test_invocation(file_trigger_groups),
        "flutter_integ_t_signal": ("Flutter" in file_trigger_groups),
        "flutter_integ_t_d": ",".join(file_flutter_devices),
    })

# Save (schema unchanged)
out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV} (rows={len(out_df)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_YML_FilesV7.0.csv (rows=12667)


In [4]:
# -*- coding: utf-8 -*-
"""
Post-process emulator coverage params.

Reads the main detector CSV (OUTPUT_CSV), inspects only rows whose
execution_environment ∈ {Generic_reactivecircus, Generic_Malinskiy, Generic_DIY},
re-opens the corresponding YAML, and outputs a CSV with booleans showing whether
each of the four emulator coverage params is explicitly defined:

  - api-level
  - arch (or ABI segment in system-images path)
  - profile (device model / AVD)
  - target (system image channel: google_apis, google_apis_playstore, default, aosp-*)

Adjust CONFIG_DIR / INPUT_CSV / OUTPUT_COVERAGE_CSV if needed.
"""

import re
import pandas as pd
from pathlib import Path
from typing import Iterable

# --- CONFIG (keep consistent with your main script) ---
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files")
INPUT_CSV  = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_YML_FilesV7.0.csv")
OUTPUT_COVERAGE_CSV = INPUT_CSV.with_name(INPUT_CSV.stem + "_emulator_coverage.csv")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_COVERAGE_CSV.parent.mkdir(parents=True, exist_ok=True)

# --- lightweight helpers copied from your scanner ---
COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

def normalize_block_keys(text: str) -> str:
    # expose run/script/command bodies; drop YAML list dashes to make ^ anchors behave
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\||>|\|\-)\s*(.+)$', r'\2', text)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(\||>|\|\-)\s*$', '', text)
    text = re.sub(r'(?m)^\s*-\s*', '', text)
    return text

def read_text_safely(p: Path) -> str:
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

# --- what we’ll treat as “emulator exec env” from your detector output ---
EMULATOR_ENVS = {"Generic_reactivecircus", "Generic_Malinskiy", "Generic_DIY"}

# --- compiled regexes to detect explicit settings ---
# api-level:
API_LEVEL_RE = re.compile(r'(?mi)^\s*api[-_ ]?level\s*:\s*\S+')
API_IN_SYSIMG_RE = re.compile(r'(?mi)system-images;android-(\d+)\b')  # sdkmanager/avdmanager path

# arch / abi:
ARCH_KEY_RE   = re.compile(r'(?mi)^\s*arch\s*:\s*(x86|x86_64|arm64|armeabi)')
ABI_KEY_RE    = re.compile(r'(?mi)^\s*abi\s*:\s*(x86|x86_64|arm64|armeabi)')
ABI_IN_SYSIMG = re.compile(r'(?mi)system-images;android-[^;\n]*;[^;\n]*;(x86|x86_64|arm64|armeabi)\b')

# profile / model / avd device hints:
PROFILE_KEY_RE   = re.compile(r'(?mi)^\s*profile\s*:\s*\S+')
DEVICE_KEY_RE    = re.compile(r'(?mi)^\s*device\s*:\s*(pixel[^\s]*)')
AVD_CREATE_D_RE  = re.compile(r'(?mi)\b(avdmanager|android)\b[^\n]*\bcreate\s+avd\b[^\n]*\b-d\s+\S+')
EMULATOR_AVD_RE  = re.compile(r'(?mi)\bemulator\b[^\n]*\s-?avd\s+\S+')

# target / system image channel:
TARGET_KEY_RE    = re.compile(r'(?mi)^\s*target\s*:\s*(google_apis|google_apis_playstore|default|aosp[^\s]*)')
TARGET_IN_SYSIMG = re.compile(r'(?mi)system-images;android-[^;\n]*;(google_apis|google_apis_playstore|default|aosp[^;]*)')

def any_match(res: Iterable[re.Pattern], text: str) -> bool:
    return any(r.search(text) for r in res)

def detect_params(yaml_text: str) -> dict:
    # normalize for robust matching
    t = normalize_block_keys(strip_comments(yaml_text))
    # API level explicit?
    api_set = any_match([API_LEVEL_RE, API_IN_SYSIMG_RE], t)
    # arch/abi explicit?
    arch_set = any_match([ARCH_KEY_RE, ABI_KEY_RE, ABI_IN_SYSIMG], t)
    # profile/device explicit?
    profile_set = any_match([PROFILE_KEY_RE, DEVICE_KEY_RE, AVD_CREATE_D_RE, EMULATOR_AVD_RE], t)
    # target/channel explicit?
    target_set = any_match([TARGET_KEY_RE, TARGET_IN_SYSIMG], t)
    return {
        "api_level_defined": bool(api_set),
        "arch_defined": bool(arch_set),
        "profile_defined": bool(profile_set),
        "target_defined": bool(target_set),
    }

# --- main ---
df = pd.read_csv(INPUT_CSV)
# keep only emulator exec envs
emu_df = df[df["execution_environment"].isin(EMULATOR_ENVS)].copy()

rows = []
for _, r in emu_df.iterrows():
    filename = str(r["filename"])
    yaml_path = CONFIG_DIR / filename
    txt = read_text_safely(yaml_path)
    params = detect_params(txt)
    rows.append({
        "filename": filename,
        "execution_environment": r["execution_environment"],
        **params,
    })

out = pd.DataFrame(rows)
out.to_csv(OUTPUT_COVERAGE_CSV, index=False, encoding="utf-8-sig")
print(f"Saved emulator coverage matrix: {OUTPUT_COVERAGE_CSV} (rows={len(out)})")


Saved emulator coverage matrix: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_YML_FilesV7.0_emulator_coverage.csv (rows=429)
